# 5. R-based Post-Processing: FastReseg & CellSPA

**FastReseg** — Post-hoc segmentation refinement (reassigns transcripts at cell boundaries)  
**CellSPA** — Segmentation quality assessment

Both are R packages. This notebook uses `rpy2` for interoperability, but they can also
be run as standalone R scripts via SLURM.

In [ ]:
import os
import yaml
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline

In [ ]:
CONFIG_PATH = "../config/pipeline_config.yaml"
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

SAMPLE_ID = cfg["sample"]["id"]
BASE_OUT = Path(cfg["paths"]["base_output_dir"]) / SAMPLE_ID

## Set up R interop

In [ ]:
from rpy2.robjects import r, pandas2ri
from rpy2.robjects import globalenv
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects.conversion import localconverter

%load_ext rpy2.ipython

print("R version:")
%R R.version.string

---
## Part 1: FastReseg

FastReseg refines cell boundaries by re-assigning transcripts near cell borders using
a statistical model. It works best when you have:
- Transcript coordinates with initial cell assignments
- Reference gene expression profiles per cell type

See: [FastReseg GitHub](https://github.com/Nanostring-Biostats/FastReseg)

In [ ]:
%%R
library(FastReseg)
cat("FastReseg loaded, version:", as.character(packageVersion("FastReseg")), "\n")

In [ ]:
# Choose which segmentation method's results to refine
SOURCE_METHOD = "proseg"  # or "baysor", "cellpose"
source_dir = BASE_OUT / SOURCE_METHOD
h5ad_path = source_dir / f"{SAMPLE_ID}.h5ad"

if h5ad_path.exists():
    adata = sc.read_h5ad(h5ad_path)
    print(f"Loaded {SOURCE_METHOD}: {adata.n_obs} cells × {adata.n_vars} genes")
else:
    print(f"No h5ad found at {h5ad_path}. Run {SOURCE_METHOD} first.")

In [ ]:
%%R

# FastReseg workflow template
# Adapt this based on your data structure:
#
# 1. Load transcript-level data with cell assignments
# transcript_df <- read.csv("path/to/transcripts_with_cells.csv")
#
# 2. Build count matrix and reference profiles
# counts <- ... 
# clust <- ...  # cluster assignments from annotation
# refProfiles <- ... # reference profiles per cluster
#
# 3. Run FastReseg
# result <- fastReseg_full_pipeline(
#   counts = counts,
#   clust = clust,  
#   refProfiles = refProfiles,
#   score_baseline = score_baseline,
#   lowerCutoff = 0.5,
#   higherCutoff = 0.9,
#   transcript_df = transcript_df,
#   transID_coln = "transcript_id",
#   transGene_coln = "feature_name",
#   cellID_coln = "cell_id",
#   spatLocs_colns = c("x_location", "y_location"),
#   return_perCell_results = TRUE
# )

cat("FastReseg template ready.\n")
cat("Requires: transcript-level data with cell assignments + reference profiles.\n")
cat("Run annotation/clustering first, then come back here.\n")

---
## Part 2: CellSPA — Segmentation Quality Assessment

CellSPA provides spatial metrics for evaluating segmentation quality.

See: [CellSPA GitHub](https://github.com/SydneyBioX/CellSPA)

In [ ]:
%%R
library(CellSPA)
cat("CellSPA loaded, version:", as.character(packageVersion("CellSPA")), "\n")

In [ ]:
%%R

# CellSPA QC workflow template
# Requires cell boundaries (polygons) and transcript locations
#
# cellspa_obj <- createCellSPA(
#   cell_boundaries = cell_polygons,  # sf object with cell polygons
#   transcripts = transcript_df,       # data.frame with x, y, gene columns
#   cell_metadata = cell_metadata      # optional metadata per cell
# )
#
# # Compute QC metrics
# cellspa_obj <- computeQCMetrics(cellspa_obj)
#
# # Visualize
# plotCellSPA(cellspa_obj, metric = "transcript_density")
# plotCellSPA(cellspa_obj, metric = "compactness")
# plotCellSPA(cellspa_obj, metric = "contamination")

cat("CellSPA template ready.\n")
cat("Requires: cell boundary polygons + transcript coordinates.\n")

---

## Notes

- Both FastReseg and CellSPA can be run as SLURM jobs. See `scripts/python/run_fastreseg.py`.
- For CellSPA, the SLURM QC job (`run_qc.py`) handles the Python-side metrics. 
  Use this notebook for the R-specific CellSPA spatial metrics.
- FastReseg requires cell type annotations to be effective — run annotation first,
  then come back here.